# 01 Data Inspection and Exploratory Data Analysis

This notebook is part of a reproducible machine learning project using the BRFSS 2015 diabetes health indicators dataset. All outputs are saved automatically to the `results` directory.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "code"))

from project_utils import *
set_reproducibility()

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")

Project root: /Users/guuuuu/Desktop/final0519
Dataset path: /Users/guuuuu/Desktop/final0519/diabetes_012_health_indicators_BRFSS2015.csv


## Purpose

This notebook inspects the raw BRFSS 2015 dataset before modeling. We examine data dimensions, variable types, missing values, duplicate rows, class imbalance, and important descriptive relationships. In academic machine learning, this step is essential because model performance depends strongly on data quality and the distribution of the target variable.

In [2]:
print("Loading raw data...")
raw_df = load_raw_data()
print(f"Raw dataset shape: {raw_df.shape}")
display(raw_df.head())

Loading raw data...
Raw dataset shape: (253680, 22)


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


## Data Quality Summary

The following tables document missing values, duplicate records, descriptive statistics, and the target distribution. These artifacts are saved as both CSV and XLSX files for reporting.

In [3]:
descriptive_tables(raw_df)
summary = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Missing cells", "Duplicate rows"],
    "Value": [raw_df.shape[0], raw_df.shape[1], int(raw_df.isna().sum().sum()), int(raw_df.duplicated().sum())]
})
save_table(summary, "eda_dataset_summary")
display(summary)
display(pd.read_csv(TABLES_DIR / "class_distribution.csv"))

,Metric,Value
0,Rows,253680
1,Columns,22
2,Missing cells,0
3,Duplicate rows,23899


,Class,Count,Percent
0,No diabetes,213703,84.241170
1,Prediabetes,4631,1.825528
2,Diabetes,35346,13.933302


## Exploratory Visualizations

The plots below use a restrained Morandi color palette, high-resolution export settings, and academic labels. The target distribution plot highlights the major class imbalance: prediabetes is much less frequent than the other two classes.

In [4]:
print("Generating EDA figures...")
plot_eda(raw_df)
print(f"Figures saved to: {FIGURES_DIR}")

Generating EDA figures...


Figures saved to: /Users/guuuuu/Desktop/final0519/results/figures


## Cleaned Dataset

Duplicate rows are removed for the modeling workflow. Removing exact duplicate rows reduces repeated observations that can bias training and evaluation. No missing-value imputation is required because the dataset contains no missing cells.

In [5]:
clean_df = clean_data(raw_df, remove_duplicates=True)
save_table(pd.DataFrame({
    "Metric": ["Rows before cleaning", "Rows after duplicate removal", "Removed duplicate rows"],
    "Value": [len(raw_df), len(clean_df), len(raw_df) - len(clean_df)]
}), "cleaning_summary")
clean_df.to_csv(INTERIM_DIR / "cleaned_brfss2015.csv", index=False)
print(f"Cleaned dataset shape: {clean_df.shape}")
display(clean_df.head())

Cleaned dataset shape: (229781, 22)


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


## Academic Interpretation

The BRFSS target variable is highly imbalanced, with most participants belonging to the no-diabetes group and a much smaller prediabetes group. This imbalance means that accuracy alone can be misleading: a model may appear strong by favoring the majority class. Therefore, later notebooks emphasize macro-averaged precision, recall, F1-score, ROC-AUC, and precision-recall curves.